# Milan mobile traffic forecasting - full pipeline (Kaggle / Colab)

Runs every stage of the project on the real Telecom Italia data:
ingestion -> EDA -> time-series analysis -> baselines -> tuning rounds -> final test runs -> comparison -> report tables.

**Kaggle**: attach a dataset containing the 62 `sms-call-internet-mi-YYYY-MM-DD.txt` files
(search Kaggle for "Telecom Italia Milan" / "mobile phone activity") and set `RAW_DIR` below to its input path.
Enable a GPU accelerator for the LSTM rounds if available (`Settings > Accelerator`).

**Colab**: download the files once from https://doi.org/10.7910/DVN/EGZHFV (guestbook required) into Google Drive,
mount Drive and point `RAW_DIR` at that folder.

In [ ]:
import os, sys, subprocess, pathlib

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_KAGGLE:
    # adjust to the dataset you attached
    RAW_DIR = "/kaggle/input/telecom-italia-milan"
    WORK = "/kaggle/working"
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW_DIR = "/content/drive/MyDrive/milan_raw"
    WORK = "/content"
else:
    RAW_DIR = "data/raw"
    WORK = "."
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| raw dir:", RAW_DIR)

In [ ]:
%cd {WORK}
if not pathlib.Path("Time-Series-Forecasting").exists():
    !git clone -q https://github.com/Samkwizera/Time-Series-Forecasting.git
%cd Time-Series-Forecasting
!pip install -q -r requirements.txt
!pip install -q -e .

Point the config at the raw files and keep the heavy intermediates in the writable working directory.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config/default.yaml"))
cfg["paths"]["raw_dir"] = RAW_DIR
yaml.safe_dump(cfg, open("config/kaggle.yaml", "w"), sort_keys=False)
os.environ["MILAN_CONFIG"] = "config/kaggle.yaml"
!python scripts/00_download.py --verify

## Stage 1 - ingestion (streams 20.8 GB of TSV into ~1 GB of Parquet; peak RSS is logged)

In [ ]:
!python scripts/01_ingest.py
import pandas as pd
pd.read_csv("reports/tables/memory_log.csv").tail(8)

## Stages 2-3 - exploratory and time-series analysis

In [ ]:
!python scripts/02_eda.py
!python scripts/03_tsa.py
from IPython.display import Image, display
for f in ["eda_citywide_hourly", "eda_daily_profiles", "eda_spatial_internet", "eda_clusters_internet", "tsa_acf_citywide"]:
    display(Image(f"reports/figures/{f}.png", width=900))

## Stages 4-5 - baselines and tuning rounds

Each round is defined in `experiments/tuning_plan.yaml` with the reasoning written before the run.
Results accumulate in `experiments/experiment_log.md`. Add or edit rounds after reading the log - that is the iterative loop.

In [ ]:
!python scripts/04_train.py --model baselines --part val
!python scripts/04_train.py --model baselines --part test
!python scripts/05_tune.py --model sarima
!python scripts/05_tune.py --model lightgbm --optuna 20
!python scripts/05_tune.py --model lstm
print(open("experiments/experiment_log.md").read())

## Stage 6 - final test runs with the best validation configuration and comparison

In [ ]:
# run_all.sh contains the same selection logic; here it is done explicitly so the choice is visible
import json, glob
def best(model):
    runs = [json.load(open(p)) for p in glob.glob(f"experiments/runs/{model}/*_val.json")]
    r = min(runs, key=lambda r: r["metrics"]["mase"])
    return r["run_id"].removesuffix("_val"), r["params"]
runs = {m: best(m) for m in ["sarima", "lightgbm", "lstm"]}
for m, (rid, params) in runs.items():
    print(m, rid, {k: v for k, v in params.items() if k in ("order","seasonal_order","fourier_k","lags","num_leaves","learning_rate","cell","hidden_size","num_layers","input_window")})
for m, (rid, params) in runs.items():
    cell = params.get("cell", m) if m == "lstm" else m
    subprocess.run([sys.executable, "scripts/04_train.py", "--model", cell, "--part", "test", "--run", rid,
                    "--params", json.dumps(params), "--note", "final test run of best validation config"], check=True)
lstm_cell = runs["lstm"][1].get("cell", "lstm")
!python scripts/06_compare.py --runs sarima={runs["sarima"][0]} lightgbm={runs["lightgbm"][0]} {lstm_cell}={runs["lstm"][0]}
pd.read_csv("reports/tables/results_summary.csv")

In [ ]:
for f in ["results_mase_by_horizon_and_cell", "results_forecasts_h1", "failure_daily_error", "failure_hour_daytype"]:
    display(Image(f"reports/figures/{f}.png", width=900))

## Stage 7 - tables and numbers for the report, then download `reports/` and `experiments/`

In [ ]:
!python scripts/07_report_assets.py
!zip -qr results_bundle.zip reports experiments report/generated
print("download results_bundle.zip and unpack it in your local clone, then run report/build.sh")